# WC_BADGE_DETAILS_D ETL - ODI to Databricks Migration
### Badge Details Dimension
**Source Table:** `workspace.PRXBI_TS.WC_MERCURY_BADGE_TS`
**Target Table:** `workspace.PRXBI_DW.wc_badge_details_d`
**Detection Strategy:** NOT_EXISTS (full column CDC with 40+ columns)
**DATASOURCE_NUM_ID:** 380

#### Migration Notes
- Oracle `PRXBI_DW_SEP` mapped to `workspace.PRXBI_DW`
- Oracle `PRXBI_TS_SEP` mapped to `workspace.PRXBI_TS`
- `SYSTIMESTAMP` replaced with `CURRENT_TIMESTAMP()`
- `NVL()` replaced with `COALESCE()`
- `NVL2(x,'Y','N')` replaced with `CASE WHEN x IS NOT NULL THEN 'Y' ELSE 'N' END`
- Oracle `/*+ append */` hints and `NOLOGGING` removed
- Oracle indexes removed (Delta handles via Z-ORDER)
- `DBMS_STATS` replaced with `OPTIMIZE` + `ZORDER`
- Oracle sequences (`SEQ.NEXTVAL`) replaced with `BIGINT GENERATED ALWAYS AS IDENTITY`
- Separate UPDATE + INSERT replaced with MERGE INTO
- NULL-safe comparison uses Spark `<=>` operator
- All tables use Delta format

In [ ]:
%sql
-- Step 1: Create Widgets for ETL Parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'EOD';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '1';

## Step 2: Get ETL Parameters
Read `wc_etl_parameters` to obtain extract time windows and current ROW_WID.

In [ ]:
%sql
-- Step 2a: Create temp view for last extract time
CREATE OR REPLACE TEMP VIEW v_etl_last_extract_time AS
SELECT
  COALESCE(
    MAX(CASE WHEN PARAM_NAME = 'LAST_EXTRACT_TIME' THEN CAST(PARAM_VALUE AS TIMESTAMP) END),
    CAST('1900-01-01 00:00:00' AS TIMESTAMP)
  ) AS last_extract_time
FROM workspace.PRXBI_DW.wc_etl_parameters
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID};

In [ ]:
%sql
-- Step 2b: Create temp view for current extract time
CREATE OR REPLACE TEMP VIEW v_etl_current_extract_time AS
SELECT
  COALESCE(
    MAX(CASE WHEN PARAM_NAME = 'CURRENT_EXTRACT_TIME' THEN CAST(PARAM_VALUE AS TIMESTAMP) END),
    CURRENT_TIMESTAMP()
  ) AS current_extract_time
FROM workspace.PRXBI_DW.wc_etl_parameters
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID};

In [ ]:
%sql
-- Step 2c: Create temp view for max ROW_WID
CREATE OR REPLACE TEMP VIEW v_etl_row_wid AS
SELECT COALESCE(MAX(ROW_WID), 0) AS max_row_wid
FROM workspace.PRXBI_DW.wc_badge_details_d;

In [ ]:
%sql
-- Step 2d: Display all parameters for validation
SELECT 'last_extract_time' AS param, CAST(last_extract_time AS STRING) AS value FROM v_etl_last_extract_time
UNION ALL
SELECT 'current_extract_time' AS param, CAST(current_extract_time AS STRING) AS value FROM v_etl_current_extract_time
UNION ALL
SELECT 'max_row_wid' AS param, CAST(max_row_wid AS STRING) AS value FROM v_etl_row_wid;

## Step 3: Create C$ Staging Table
Drop and recreate the staging table `c_badge_details_stg` to hold deduplicated source records from `WC_MERCURY_BADGE_TS`.
Maps to ODI C$ table `C$_0A7SUCRIPSM1CG2656H955OU5QP`.

In [ ]:
%sql
-- Step 3a: Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badge_details_stg;

In [ ]:
%sql
-- Step 3b: Create staging table with all 37 source columns
CREATE TABLE workspace.PRXBI_DW.c_badge_details_stg (
  ID                                STRING,
  BADGELOCATION                     STRING,
  BADGETOKEN                        STRING,
  BADGEVERSION                      STRING,
  CONTACTEMAIL                      STRING,
  CONTACTFIRSTNAME                  STRING,
  CONTACTJOBTITLE                   STRING,
  CONTACTLASTNAME                   STRING,
  CONTACTPERSONRXMASTERID           STRING,
  CREATEDBYREGISTRATIONTYPE         STRING,
  CREATEDBYTYPE                     STRING,
  CULTURE                           STRING,
  CUSTOMERTYPE                      STRING,
  EVENTEDITIONGBSCODE               STRING,
  ISBADGEUPDATE                     STRING,
  MARKETINGPREFERENCESPROMPTREQU    STRING,
  ORGANISATIONCITY                  STRING,
  ORGANISATIONCOUNTRYCODE           STRING,
  ORGANISATIONDISPLAYNAME           STRING,
  ORGANISATIONRXMASTERID            STRING,
  ORGANISATIONSTATE                 STRING,
  PARTICIPATINGORGANISATIONID       STRING,
  PRODUCTCODE                       STRING,
  QRCODECONTENT                     STRING,
  REGISTRATIONID                    STRING,
  STATUS                            STRING,
  SUPPORTSTAFFCOMPANYADDRESS        STRING,
  SUPPORTSTAFFCOMPANYNAME           STRING,
  SUPPORTSTAFFMOBILEPHONE           STRING,
  SUPPORTSTAFFREPORTSTO             STRING,
  SUPPORTSTAFFSTANDS                STRING,
  SUPPORTSTAFFUSERACCESS            STRING,
  VERSIONNUMBER                     DECIMAL(10,0),
  MOBILEPHONE                       STRING,
  FIRSTSCANNEDDATE                  TIMESTAMP,
  LASTPRINTEDDATE                   TIMESTAMP,
  ACCESSVALIDITYMODIFIEDDATE        TIMESTAMP,
  CREATEDDATE                       TIMESTAMP,
  COMPANYPRODUCTCODE                STRING,
  PAYMENTSTATUS                     STRING,
  PHOTOKEY                          STRING,
  PHOTOSOURCE                       STRING,
  PHOTOSOURCETYPE                   STRING
) USING DELTA;

## Step 4: Extract & Deduplicate Source Data
Insert into staging using `ROW_NUMBER() OVER(PARTITION BY TS.ID ORDER BY TS.INT_INSERT_DATE DESC, TS.VERSIONNUMBER DESC)`
with incremental time filter from ETL parameter temp views. Only keep `RNK = 1` (latest record per badge ID).

In [ ]:
%sql
-- Step 4a: Insert deduplicated source data into staging
INSERT INTO workspace.PRXBI_DW.c_badge_details_stg
SELECT
  ID,
  BADGELOCATION,
  BADGETOKEN,
  BADGEVERSION,
  CONTACTEMAIL,
  CONTACTFIRSTNAME,
  CONTACTJOBTITLE,
  CONTACTLASTNAME,
  CONTACTPERSONRXMASTERID,
  CREATEDBYREGISTRATIONTYPE,
  CREATEDBYTYPE,
  CULTURE,
  CUSTOMERTYPE,
  EVENTEDITIONGBSCODE,
  ISBADGEUPDATE,
  MARKETINGPREFERENCESPROMPTREQU,
  ORGANISATIONCITY,
  ORGANISATIONCOUNTRYCODE,
  ORGANISATIONDISPLAYNAME,
  ORGANISATIONRXMASTERID,
  ORGANISATIONSTATE,
  PARTICIPATINGORGANISATIONID,
  PRODUCTCODE,
  QRCODECONTENT,
  REGISTRATIONID,
  STATUS,
  SUPPORTSTAFFCOMPANYADDRESS,
  SUPPORTSTAFFCOMPANYNAME,
  SUPPORTSTAFFMOBILEPHONE,
  SUPPORTSTAFFREPORTSTO,
  SUPPORTSTAFFSTANDS,
  SUPPORTSTAFFUSERACCESS,
  VERSIONNUMBER,
  MOBILEPHONE,
  FIRSTSCANNEDDATE,
  LASTPRINTEDDATE,
  ACCESSVALIDITYMODIFIEDDATE,
  CREATEDDATE,
  COMPANYPRODUCTCODE,
  PAYMENTSTATUS,
  PHOTOKEY,
  PHOTOSOURCE,
  PHOTOSOURCETYPE
FROM (
  SELECT
    TS.ID,
    TS.BADGELOCATION,
    TS.BADGETOKEN,
    TS.BADGEVERSION,
    TS.CONTACTEMAIL,
    TS.CONTACTFIRSTNAME,
    TS.CONTACTJOBTITLE,
    TS.CONTACTLASTNAME,
    TS.CONTACTPERSONRXMASTERID,
    TS.CREATEDBYREGISTRATIONTYPE,
    TS.CREATEDBYTYPE,
    TS.CULTURE,
    TS.CUSTOMERTYPE,
    TS.EVENTEDITIONGBSCODE,
    TS.ISBADGEUPDATE,
    TS.MARKETINGPREFERENCESPROMPTREQU,
    TS.ORGANISATIONCITY,
    TS.ORGANISATIONCOUNTRYCODE,
    TS.ORGANISATIONDISPLAYNAME,
    TS.ORGANISATIONRXMASTERID,
    TS.ORGANISATIONSTATE,
    TS.PARTICIPATINGORGANISATIONID,
    TS.PRODUCTCODE,
    TS.QRCODECONTENT,
    TS.REGISTRATIONID,
    TS.STATUS,
    TS.SUPPORTSTAFFCOMPANYADDRESS,
    TS.SUPPORTSTAFFCOMPANYNAME,
    TS.SUPPORTSTAFFMOBILEPHONE,
    TS.SUPPORTSTAFFREPORTSTO,
    TS.SUPPORTSTAFFSTANDS,
    TS.SUPPORTSTAFFUSERACCESS,
    TS.VERSIONNUMBER,
    TS.MOBILEPHONE,
    TS.FIRSTSCANNEDDATE,
    TS.LASTPRINTEDDATE,
    TS.ACCESSVALIDITYMODIFIEDDATE,
    TS.CREATEDDATE,
    TS.COMPANYPRODUCTCODE,
    TS.PAYMENTSTATUS,
    TS.PHOTOKEY,
    TS.PHOTOSOURCE,
    TS.PHOTOSOURCETYPE,
    ROW_NUMBER() OVER (
      PARTITION BY TS.ID
      ORDER BY TS.INT_INSERT_DATE DESC, TS.VERSIONNUMBER DESC
    ) AS RNK
  FROM workspace.PRXBI_TS.WC_MERCURY_BADGE_TS TS
  WHERE TS.INT_INSERT_DATE > (SELECT last_extract_time FROM v_etl_last_extract_time)
    AND TS.INT_INSERT_DATE <= (SELECT current_extract_time FROM v_etl_current_extract_time)
) dedup
WHERE RNK = 1;

In [ ]:
%sql
-- Step 4b: Validate staging record count
SELECT COUNT(*) AS staging_record_count FROM workspace.PRXBI_DW.c_badge_details_stg;

## Step 5: Create I$ Flow Table
Drop and recreate the flow table `i_badge_details_flow` which holds the change-detected records
with an `IND_UPDATE` flag ('I' for insert, 'U' for update).
Maps to ODI I$ table `I$_WADHBP95VCNT6J0F17HEGQQJ3GB`.

In [ ]:
%sql
-- Step 5a: Drop flow table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badge_details_flow;

In [ ]:
%sql
-- Step 5b: Create flow table with all renamed columns
CREATE TABLE workspace.PRXBI_DW.i_badge_details_flow (
  BADGE_ID                    STRING,
  BADGE_LOCATION              STRING,
  BADGE_TOKEN                 STRING,
  BADGE_VERSION               STRING,
  CONTACT_EMAIL               STRING,
  CONTACT_FIRST_NAME          STRING,
  CONTACT_JOB_TITLE           STRING,
  CONTACT_LAST_NAME           STRING,
  CONTACT_PERSON_ID           STRING,
  CREATION_REG_TYPE           STRING,
  CREATION_TYPE               STRING,
  CULTURE                     STRING,
  CUSTOMER_TYPE               STRING,
  EVENT_EDITION_CODE          STRING,
  BADGE_UPDATE_FLG            STRING,
  MARKETING_PREF_PROMPT       STRING,
  ORG_CITY                    STRING,
  ORG_COUNTRY                 STRING,
  ORG_NAME                    STRING,
  ORG_ID                      STRING,
  ORG_STATE                   STRING,
  PARTICIPATING_ORG_ID        STRING,
  PRODUCT_CODE                STRING,
  QR_CODE                     STRING,
  REGISTRATION_ID             STRING,
  STATUS                      STRING,
  STAFF_COMPANY_ADDR          STRING,
  STAFF_COMPANY_NAME          STRING,
  STAFF_PHONE_NUM             STRING,
  STAFF_REPORTING             STRING,
  STAFF_STANDS                STRING,
  STAFF_USER_ACCESS           STRING,
  VERSION_NUM                 DECIMAL(10,0),
  MOBILEPHONE                 STRING,
  FIRSTSCANNEDDATE            TIMESTAMP,
  FIRSTSCANNEDDATE_FLG        STRING,
  LASTPRINTEDDATE             TIMESTAMP,
  LASTPRINTEDDATE_FLG         STRING,
  ACCESSVALIDITYMODIFIEDDATE  TIMESTAMP,
  CREATEDDATE                 TIMESTAMP,
  COMPANYPRODUCTCODE          STRING,
  PAYMENTSTATUS               STRING,
  PHOTOKEY                    STRING,
  PHOTOSOURCE                 STRING,
  PHOTOSOURCETYPE             STRING,
  PACKAGE_NAME                STRING,
  INTEGRATION_ID              STRING,
  DATASOURCE_NUM_ID           INT,
  IND_UPDATE                  STRING
) USING DELTA;

## Step 6: Enrich & Detect Changes
Insert into the flow table with all column renames, `CASE WHEN` for FLG columns,
LEFT JOIN to `WC_BADGE_PRODUCT_D` (using `ROW_NUMBER PARTITION BY SKU ORDER BY ID DESC`, `rn=1`) for `PACKAGE_NAME`.
NOT EXISTS detection with `<=>` null-safe comparison on ALL columns against `wc_badge_details_d`.

In [ ]:
%sql
-- Step 6a: Insert into flow table with column renames, enrichment, and NOT EXISTS change detection
INSERT INTO workspace.PRXBI_DW.i_badge_details_flow
SELECT
  S.ID                                AS BADGE_ID,
  S.BADGELOCATION                     AS BADGE_LOCATION,
  S.BADGETOKEN                        AS BADGE_TOKEN,
  S.BADGEVERSION                      AS BADGE_VERSION,
  S.CONTACTEMAIL                      AS CONTACT_EMAIL,
  S.CONTACTFIRSTNAME                  AS CONTACT_FIRST_NAME,
  S.CONTACTJOBTITLE                   AS CONTACT_JOB_TITLE,
  S.CONTACTLASTNAME                   AS CONTACT_LAST_NAME,
  S.CONTACTPERSONRXMASTERID           AS CONTACT_PERSON_ID,
  S.CREATEDBYREGISTRATIONTYPE         AS CREATION_REG_TYPE,
  S.CREATEDBYTYPE                     AS CREATION_TYPE,
  S.CULTURE                           AS CULTURE,
  S.CUSTOMERTYPE                      AS CUSTOMER_TYPE,
  S.EVENTEDITIONGBSCODE               AS EVENT_EDITION_CODE,
  S.ISBADGEUPDATE                     AS BADGE_UPDATE_FLG,
  S.MARKETINGPREFERENCESPROMPTREQU    AS MARKETING_PREF_PROMPT,
  S.ORGANISATIONCITY                  AS ORG_CITY,
  S.ORGANISATIONCOUNTRYCODE           AS ORG_COUNTRY,
  S.ORGANISATIONDISPLAYNAME           AS ORG_NAME,
  S.ORGANISATIONRXMASTERID            AS ORG_ID,
  S.ORGANISATIONSTATE                 AS ORG_STATE,
  S.PARTICIPATINGORGANISATIONID       AS PARTICIPATING_ORG_ID,
  S.PRODUCTCODE                       AS PRODUCT_CODE,
  S.QRCODECONTENT                     AS QR_CODE,
  S.REGISTRATIONID                    AS REGISTRATION_ID,
  S.STATUS                            AS STATUS,
  S.SUPPORTSTAFFCOMPANYADDRESS        AS STAFF_COMPANY_ADDR,
  S.SUPPORTSTAFFCOMPANYNAME           AS STAFF_COMPANY_NAME,
  S.SUPPORTSTAFFMOBILEPHONE           AS STAFF_PHONE_NUM,
  S.SUPPORTSTAFFREPORTSTO             AS STAFF_REPORTING,
  S.SUPPORTSTAFFSTANDS                AS STAFF_STANDS,
  S.SUPPORTSTAFFUSERACCESS            AS STAFF_USER_ACCESS,
  S.VERSIONNUMBER                     AS VERSION_NUM,
  S.MOBILEPHONE                       AS MOBILEPHONE,
  S.FIRSTSCANNEDDATE                  AS FIRSTSCANNEDDATE,
  CASE WHEN S.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
  S.LASTPRINTEDDATE                   AS LASTPRINTEDDATE,
  CASE WHEN S.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END  AS LASTPRINTEDDATE_FLG,
  S.ACCESSVALIDITYMODIFIEDDATE        AS ACCESSVALIDITYMODIFIEDDATE,
  S.CREATEDDATE                       AS CREATEDDATE,
  S.COMPANYPRODUCTCODE                AS COMPANYPRODUCTCODE,
  S.PAYMENTSTATUS                     AS PAYMENTSTATUS,
  S.PHOTOKEY                          AS PHOTOKEY,
  S.PHOTOSOURCE                       AS PHOTOSOURCE,
  S.PHOTOSOURCETYPE                   AS PHOTOSOURCETYPE,
  PKG.NAME                            AS PACKAGE_NAME,
  S.ID                                AS INTEGRATION_ID,
  380                                 AS DATASOURCE_NUM_ID,
  'I'                                 AS IND_UPDATE
FROM workspace.PRXBI_DW.c_badge_details_stg S
LEFT OUTER JOIN (
  SELECT SKU, NAME
  FROM (
    SELECT
      SKU,
      NAME,
      ROW_NUMBER() OVER (PARTITION BY SKU ORDER BY ID DESC) AS rn
    FROM workspace.PRXBI_DW.wc_badge_product_d
  ) ranked
  WHERE rn = 1
) PKG
  ON S.PRODUCTCODE = PKG.SKU
WHERE NOT EXISTS (
  SELECT 1
  FROM workspace.PRXBI_DW.wc_badge_details_d T
  WHERE T.INTEGRATION_ID      <=> S.ID
    AND T.DATASOURCE_NUM_ID   <=> 380
    AND T.BADGE_ID            <=> S.ID
    AND T.BADGE_LOCATION      <=> S.BADGELOCATION
    AND T.BADGE_TOKEN         <=> S.BADGETOKEN
    AND T.BADGE_VERSION       <=> S.BADGEVERSION
    AND T.CONTACT_EMAIL       <=> S.CONTACTEMAIL
    AND T.CONTACT_FIRST_NAME  <=> S.CONTACTFIRSTNAME
    AND T.CONTACT_JOB_TITLE   <=> S.CONTACTJOBTITLE
    AND T.CONTACT_LAST_NAME   <=> S.CONTACTLASTNAME
    AND T.CONTACT_PERSON_ID   <=> S.CONTACTPERSONRXMASTERID
    AND T.CREATION_REG_TYPE   <=> S.CREATEDBYREGISTRATIONTYPE
    AND T.CREATION_TYPE       <=> S.CREATEDBYTYPE
    AND T.CULTURE             <=> S.CULTURE
    AND T.CUSTOMER_TYPE       <=> S.CUSTOMERTYPE
    AND T.EVENT_EDITION_CODE  <=> S.EVENTEDITIONGBSCODE
    AND T.BADGE_UPDATE_FLG    <=> S.ISBADGEUPDATE
    AND T.MARKETING_PREF_PROMPT <=> S.MARKETINGPREFERENCESPROMPTREQU
    AND T.ORG_CITY            <=> S.ORGANISATIONCITY
    AND T.ORG_COUNTRY         <=> S.ORGANISATIONCOUNTRYCODE
    AND T.ORG_NAME            <=> S.ORGANISATIONDISPLAYNAME
    AND T.ORG_ID              <=> S.ORGANISATIONRXMASTERID
    AND T.ORG_STATE           <=> S.ORGANISATIONSTATE
    AND T.PARTICIPATING_ORG_ID <=> S.PARTICIPATINGORGANISATIONID
    AND T.PRODUCT_CODE        <=> S.PRODUCTCODE
    AND T.QR_CODE             <=> S.QRCODECONTENT
    AND T.REGISTRATION_ID     <=> S.REGISTRATIONID
    AND T.STATUS              <=> S.STATUS
    AND T.STAFF_COMPANY_ADDR  <=> S.SUPPORTSTAFFCOMPANYADDRESS
    AND T.STAFF_COMPANY_NAME  <=> S.SUPPORTSTAFFCOMPANYNAME
    AND T.STAFF_PHONE_NUM     <=> S.SUPPORTSTAFFMOBILEPHONE
    AND T.STAFF_REPORTING     <=> S.SUPPORTSTAFFREPORTSTO
    AND T.STAFF_STANDS        <=> S.SUPPORTSTAFFSTANDS
    AND T.STAFF_USER_ACCESS   <=> S.SUPPORTSTAFFUSERACCESS
    AND T.VERSION_NUM         <=> S.VERSIONNUMBER
    AND T.MOBILEPHONE         <=> S.MOBILEPHONE
    AND T.FIRSTSCANNEDDATE    <=> S.FIRSTSCANNEDDATE
    AND T.LASTPRINTEDDATE     <=> S.LASTPRINTEDDATE
    AND T.ACCESSVALIDITYMODIFIEDDATE <=> S.ACCESSVALIDITYMODIFIEDDATE
    AND T.CREATEDDATE         <=> S.CREATEDDATE
    AND T.COMPANYPRODUCTCODE  <=> S.COMPANYPRODUCTCODE
    AND T.PAYMENTSTATUS       <=> S.PAYMENTSTATUS
    AND T.PHOTOKEY            <=> S.PHOTOKEY
    AND T.PHOTOSOURCE         <=> S.PHOTOSOURCE
    AND T.PHOTOSOURCETYPE     <=> S.PHOTOSOURCETYPE
);

In [ ]:
%sql
-- Step 6b: Validate flow table record count
SELECT COUNT(*) AS flow_record_count FROM workspace.PRXBI_DW.i_badge_details_flow;

## Step 7: Flag Updates
Set `IND_UPDATE = 'U'` for records in the flow table where a matching `(INTEGRATION_ID, DATASOURCE_NUM_ID)`
already exists in the target table. These records will be applied as updates rather than inserts.

In [ ]:
%sql
-- Step 7a: Flag records for update where they already exist in target
UPDATE workspace.PRXBI_DW.i_badge_details_flow F
SET IND_UPDATE = 'U'
WHERE EXISTS (
  SELECT 1
  FROM workspace.PRXBI_DW.wc_badge_details_d T
  WHERE T.INTEGRATION_ID = F.INTEGRATION_ID
    AND T.DATASOURCE_NUM_ID = F.DATASOURCE_NUM_ID
);

In [ ]:
%sql
-- Step 7b: Validate IND_UPDATE breakdown
SELECT IND_UPDATE, COUNT(*) AS record_count
FROM workspace.PRXBI_DW.i_badge_details_flow
GROUP BY IND_UPDATE
ORDER BY IND_UPDATE;

## Step 8: MERGE to Target
MERGE replaces the separate UPDATE + INSERT from ODI.
- **WHEN MATCHED AND IND_UPDATE = 'U':** Update all columns in the target + `W_UPDATE_DT = CURRENT_TIMESTAMP()`.
- **WHEN NOT MATCHED AND IND_UPDATE = 'I':** Insert new records (ROW_WID generated by IDENTITY column) + `W_INSERT_DT`, `W_UPDATE_DT`.

In [ ]:
%sql
-- Step 8: MERGE INTO target from flow table
MERGE INTO workspace.PRXBI_DW.wc_badge_details_d T
USING workspace.PRXBI_DW.i_badge_details_flow S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
  AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID

WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
  T.BADGE_ID                    = S.BADGE_ID,
  T.BADGE_LOCATION              = S.BADGE_LOCATION,
  T.BADGE_TOKEN                 = S.BADGE_TOKEN,
  T.BADGE_VERSION               = S.BADGE_VERSION,
  T.CONTACT_EMAIL               = S.CONTACT_EMAIL,
  T.CONTACT_FIRST_NAME          = S.CONTACT_FIRST_NAME,
  T.CONTACT_JOB_TITLE           = S.CONTACT_JOB_TITLE,
  T.CONTACT_LAST_NAME           = S.CONTACT_LAST_NAME,
  T.CONTACT_PERSON_ID           = S.CONTACT_PERSON_ID,
  T.CREATION_REG_TYPE           = S.CREATION_REG_TYPE,
  T.CREATION_TYPE               = S.CREATION_TYPE,
  T.CULTURE                     = S.CULTURE,
  T.CUSTOMER_TYPE               = S.CUSTOMER_TYPE,
  T.EVENT_EDITION_CODE          = S.EVENT_EDITION_CODE,
  T.BADGE_UPDATE_FLG            = S.BADGE_UPDATE_FLG,
  T.MARKETING_PREF_PROMPT       = S.MARKETING_PREF_PROMPT,
  T.ORG_CITY                    = S.ORG_CITY,
  T.ORG_COUNTRY                 = S.ORG_COUNTRY,
  T.ORG_NAME                    = S.ORG_NAME,
  T.ORG_ID                      = S.ORG_ID,
  T.ORG_STATE                   = S.ORG_STATE,
  T.PARTICIPATING_ORG_ID        = S.PARTICIPATING_ORG_ID,
  T.PRODUCT_CODE                = S.PRODUCT_CODE,
  T.QR_CODE                     = S.QR_CODE,
  T.REGISTRATION_ID             = S.REGISTRATION_ID,
  T.STATUS                      = S.STATUS,
  T.STAFF_COMPANY_ADDR          = S.STAFF_COMPANY_ADDR,
  T.STAFF_COMPANY_NAME          = S.STAFF_COMPANY_NAME,
  T.STAFF_PHONE_NUM             = S.STAFF_PHONE_NUM,
  T.STAFF_REPORTING             = S.STAFF_REPORTING,
  T.STAFF_STANDS                = S.STAFF_STANDS,
  T.STAFF_USER_ACCESS           = S.STAFF_USER_ACCESS,
  T.VERSION_NUM                 = S.VERSION_NUM,
  T.MOBILEPHONE                 = S.MOBILEPHONE,
  T.FIRSTSCANNEDDATE            = S.FIRSTSCANNEDDATE,
  T.FIRSTSCANNEDDATE_FLG        = S.FIRSTSCANNEDDATE_FLG,
  T.LASTPRINTEDDATE             = S.LASTPRINTEDDATE,
  T.LASTPRINTEDDATE_FLG         = S.LASTPRINTEDDATE_FLG,
  T.ACCESSVALIDITYMODIFIEDDATE  = S.ACCESSVALIDITYMODIFIEDDATE,
  T.CREATEDDATE                 = S.CREATEDDATE,
  T.COMPANYPRODUCTCODE          = S.COMPANYPRODUCTCODE,
  T.PAYMENTSTATUS               = S.PAYMENTSTATUS,
  T.PHOTOKEY                    = S.PHOTOKEY,
  T.PHOTOSOURCE                 = S.PHOTOSOURCE,
  T.PHOTOSOURCETYPE             = S.PHOTOSOURCETYPE,
  T.PACKAGE_NAME                = S.PACKAGE_NAME,
  T.W_UPDATE_DT                 = CURRENT_TIMESTAMP()

WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
  BADGE_ID,
  BADGE_LOCATION,
  BADGE_TOKEN,
  BADGE_VERSION,
  CONTACT_EMAIL,
  CONTACT_FIRST_NAME,
  CONTACT_JOB_TITLE,
  CONTACT_LAST_NAME,
  CONTACT_PERSON_ID,
  CREATION_REG_TYPE,
  CREATION_TYPE,
  CULTURE,
  CUSTOMER_TYPE,
  EVENT_EDITION_CODE,
  BADGE_UPDATE_FLG,
  MARKETING_PREF_PROMPT,
  ORG_CITY,
  ORG_COUNTRY,
  ORG_NAME,
  ORG_ID,
  ORG_STATE,
  PARTICIPATING_ORG_ID,
  PRODUCT_CODE,
  QR_CODE,
  REGISTRATION_ID,
  STATUS,
  STAFF_COMPANY_ADDR,
  STAFF_COMPANY_NAME,
  STAFF_PHONE_NUM,
  STAFF_REPORTING,
  STAFF_STANDS,
  STAFF_USER_ACCESS,
  VERSION_NUM,
  MOBILEPHONE,
  FIRSTSCANNEDDATE,
  FIRSTSCANNEDDATE_FLG,
  LASTPRINTEDDATE,
  LASTPRINTEDDATE_FLG,
  ACCESSVALIDITYMODIFIEDDATE,
  CREATEDDATE,
  COMPANYPRODUCTCODE,
  PAYMENTSTATUS,
  PHOTOKEY,
  PHOTOSOURCE,
  PHOTOSOURCETYPE,
  PACKAGE_NAME,
  INTEGRATION_ID,
  DATASOURCE_NUM_ID,
  W_INSERT_DT,
  W_UPDATE_DT
) VALUES (
  S.BADGE_ID,
  S.BADGE_LOCATION,
  S.BADGE_TOKEN,
  S.BADGE_VERSION,
  S.CONTACT_EMAIL,
  S.CONTACT_FIRST_NAME,
  S.CONTACT_JOB_TITLE,
  S.CONTACT_LAST_NAME,
  S.CONTACT_PERSON_ID,
  S.CREATION_REG_TYPE,
  S.CREATION_TYPE,
  S.CULTURE,
  S.CUSTOMER_TYPE,
  S.EVENT_EDITION_CODE,
  S.BADGE_UPDATE_FLG,
  S.MARKETING_PREF_PROMPT,
  S.ORG_CITY,
  S.ORG_COUNTRY,
  S.ORG_NAME,
  S.ORG_ID,
  S.ORG_STATE,
  S.PARTICIPATING_ORG_ID,
  S.PRODUCT_CODE,
  S.QR_CODE,
  S.REGISTRATION_ID,
  S.STATUS,
  S.STAFF_COMPANY_ADDR,
  S.STAFF_COMPANY_NAME,
  S.STAFF_PHONE_NUM,
  S.STAFF_REPORTING,
  S.STAFF_STANDS,
  S.STAFF_USER_ACCESS,
  S.VERSION_NUM,
  S.MOBILEPHONE,
  S.FIRSTSCANNEDDATE,
  S.FIRSTSCANNEDDATE_FLG,
  S.LASTPRINTEDDATE,
  S.LASTPRINTEDDATE_FLG,
  S.ACCESSVALIDITYMODIFIEDDATE,
  S.CREATEDDATE,
  S.COMPANYPRODUCTCODE,
  S.PAYMENTSTATUS,
  S.PHOTOKEY,
  S.PHOTOSOURCE,
  S.PHOTOSOURCETYPE,
  S.PACKAGE_NAME,
  S.INTEGRATION_ID,
  S.DATASOURCE_NUM_ID,
  CURRENT_TIMESTAMP(),
  CURRENT_TIMESTAMP()
);

## Step 9: Optimize & Cleanup
Replace Oracle `DBMS_STATS.GATHER_TABLE_STATS` with Delta `OPTIMIZE` and `ZORDER BY` for query performance.
Drop the staging (C$) and flow (I$) tables used during the ETL process.

In [ ]:
%sql
-- Step 9a: Optimize target table with ZORDER for frequently filtered columns
OPTIMIZE workspace.PRXBI_DW.wc_badge_details_d
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

-- Step 9b: Drop staging table
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badge_details_stg;

-- Step 9c: Drop flow table
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badge_details_flow;

## Step 10: Validation
Verify the target table record count and inspect sample records to confirm the load completed successfully.

In [ ]:
%sql
-- Step 10: Final validation
SELECT
  COUNT(*)                        AS total_records,
  COUNT(DISTINCT INTEGRATION_ID)  AS distinct_badges,
  MIN(W_INSERT_DT)                AS earliest_insert,
  MAX(W_UPDATE_DT)                AS latest_update,
  SUM(CASE WHEN W_UPDATE_DT > W_INSERT_DT THEN 1 ELSE 0 END) AS updated_records
FROM workspace.PRXBI_DW.wc_badge_details_d
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID};

## Conversion Notes (ODI to Databricks)

| ODI / Oracle Construct | Databricks / Spark SQL Equivalent |
|---|---|
| `PRXBI_DW_SEP` schema | `workspace.PRXBI_DW` |
| `PRXBI_TS_SEP` schema | `workspace.PRXBI_TS` |
| `SYSTIMESTAMP` | `CURRENT_TIMESTAMP()` |
| `NVL(col, val)` | `COALESCE(col, val)` |
| `NVL2(x,'Y','N')` | `CASE WHEN x IS NOT NULL THEN 'Y' ELSE 'N' END` |
| `/*+ append */` hint | Removed (Delta handles append natively) |
| `NOLOGGING` | Removed (Delta manages transaction logs) |
| Oracle indexes | Removed (Delta uses Z-ORDER for data skipping) |
| `DBMS_STATS.GATHER_TABLE_STATS` | `OPTIMIZE` + `ZORDER BY` |
| `WC_BADGE_DETAILS_D_SEQ.NEXTVAL` for ROW_WID | `BIGINT GENERATED ALWAYS AS IDENTITY` on target table |
| `#GLOBAL.v_ETL_JOB_TYPE` | Widget `${ETL_JOB_TYPE}` |
| `#GLOBAL.V_ETL_LAST_EXTRACT_TIME` | Temp view `v_etl_last_extract_time` |
| `#GLOBAL.V_ETL_CURRENT_EXTRACT_TIME` | Temp view `v_etl_current_extract_time` |
| `TO_TIMESTAMP('#GLOBAL.V_ETL_LAST_EXTRACT_TIME','YYYY-MM-DD HH24:MI:SS.FF')` | Subselect from temp view |
| Separate UPDATE + INSERT | `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` |
| `VARCHAR2` | `STRING` |
| `NUMBER(x,y)` | `DECIMAL(x,y)` or `BIGINT`/`INT` |
| `TIMESTAMP(6)` / `TIMESTAMP(7)` | `TIMESTAMP` |
| `(T.COL = S.COL) OR (T.COL IS NULL AND S.COL IS NULL)` | `T.COL <=> S.COL` (NULL-safe equality) |
| C$ table `C$_0A7SUCRIPSM1CG2656H955OU5QP` | `workspace.PRXBI_DW.c_badge_details_stg` (Delta) |
| I$ table `I$_WADHBP95VCNT6J0F17HEGQQJ3GB` | `workspace.PRXBI_DW.i_badge_details_flow` (Delta) |
| `INNER JOIN` subquery with `MAX(INT_INSERT_DATE)` + `MAX(VERSIONNUMBER)` | `ROW_NUMBER() OVER(PARTITION BY ID ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC)` with `RNK=1` |
| `LEFT OUTER JOIN WC_BADGE_PRODUCT_D` with `RANK()` | `LEFT OUTER JOIN` with `ROW_NUMBER() OVER(PARTITION BY SKU ORDER BY ID DESC)` where `rn=1` |